# Silver → Gold

## Objetivo

Modelar a camada Gold em Star Schema para consumo analítico, gerar a tabela de contexto para o assistente de IA (RAG) e validar a modelagem respondendo às 6 perguntas de negócio.

## Tabelas geradas

- **Dimensões:** `dim_movies`, `dim_genres`, `dim_people`, `dim_companies`, `dim_reviews`
- **Pontes (relações N:N):** `bridge_movie_genre`, `bridge_movie_person`, `bridge_movie_company`
- **Fato:** `fact_movies_performance`
- **IA:** `gold_genai_movies_context`

## Decisões de modelagem

- A `dim_movies` vem de `silver.tb_info_filmes`. A fato parte da `dim_movies` (filmes lançados) e usa `left join` com financeiro e métricas, garantindo um registro por filme e nenhuma FK nula.
- Os 1.264 ids que existem em financeiro e métricas, mas não em `tb_info_filmes`, ficam fora da Gold. Sem título, data ou status não há como ligá-los à `dim_movies`. Eles representam cerca de 0,21% da receita em R$.
- As chaves substitutas usam `row_number()` ordenado pela chave natural, para serem idênticas a cada execução. O `monotonically_increasing_id()` não é determinístico e mudaria as chaves a cada reprocessamento.
- Filme lançado é o filme com `status_filme = 'Lançado'`.
- As tabelas são gravadas com `overwrite`, pela mesma razão da Silver: a Gold é recalculada por completo a cada execução.
- A ordem das células importa. As dimensões são gravadas primeiro, e as pontes, a fato e a tabela de IA leem as chaves substitutas das dimensões já gravadas.

## Configuração

Nomes do catalog e dos schemas centralizados em variáveis, e imports usados em todo o notebook.

In [0]:
catalog = "cineData_analytics"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import LongType, IntegerType, DoubleType, DecimalType

## 1. dim_movies

Metadados descritivos de cada filme. Origem: `silver.tb_info_filmes`.

**Decisões:**
- Contém todos os filmes da Silver, independentemente do status. O filtro de filmes lançados é aplicado na fato e nas consultas.
- `id_filme` é armazenado como `STRING` (chave natural), e a chave de junção é a `sk_movie_id`.
- Colunas descritivas que não fazem parte do contrato da dimensão (como `titulo_original` e `frase_divulgacao`) não são levadas para a Gold.

In [0]:
df_info_filmes = spark.table(f"{silver_schema}.tb_info_filmes")

df_dim_movies = (
    df_info_filmes

    #chave substituta gerada com row_number ordenado pela chave natural 
    .withColumn("sk_movie_id", F.row_number().over(Window.orderBy("id_filme")).cast("bigint"))

    #a chave natural é armazenada como string
    .withColumn("id_filme", F.col("id_filme").cast("string"))

    .select(
        "sk_movie_id",
        "id_filme",
        "titulo",
        "data_lancamento",
        "ano_lancamento",
        "duracao_minutos",
        "idioma_original",
        "status_filme",
        "sinopse"
    )
)

df_dim_movies.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_movies")

display(df_dim_movies.limit(25))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sk_movie_id,id_filme,titulo,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse
1,14564,Rings,2017-02-01,2017,102,en,Lançado,"\Julia becomes worried about her boyfriend Holt when he explores the dark urban legend of a mysterious videotape said to kill the watcher seven days after viewing. She sacrifices herself to save her boyfriend and in doing so makes a horrifying discovery: there is a """"\""""movie within the movie""""\"""" that no one has ever seen before.""""""""First you watch it. Then you die."
2,32471,Mixtape,2021-12-03,2021,94,en,Lançado,null
3,38258,Grizzly II: Revenge,2020-02-17,2020,74,en,Lançado,\All hell breaks loose when a giant grizzly
4,38492,Billy Joel - Live at Yankee Stadium,2022-06-22,2022,86,en,Lançado,Billy Joel plays his greatest hits in the Big Apple.
5,38700,Bad Boys for Life,2020-01-15,2020,124,en,Lançado,"Marcus and Mike are forced to confront new threats, career changes, and midlife crises as they join the newly created elite team AMMO of the Miami police department to take down the ruthless Armando Armas, the vicious leader of a Miami drug cartel."
6,42018,The Horse Thief,2019-03-19,2019,88,zh,Lançado,"Devout Buddhists, Norbu and Dolma live with their young son Tashi in a clan in Tibet. Norbu is a highwayman. After Norbu is charged with stealing from the temple, he and his family are banished. Impoverished and marginalized, they can do little when their beloved son becomes ill. Tashi dies of a fever. After a second son is born, Norbu focuses his every action on keeping this child alive, seeking re-admission to the clan for his wife and child, then risking all to save them from isolation and starvation in winter."
7,42330,Monkey Magic,2018-09-22,2018,66,zh,Lançado,"Dearth Voyd, the ruler of the dark side of the universe, wants to turn the human world into a place of violence and evil!! However, a huge meteor falls upon Flower-Fruit mountain giving birth to Kongo, a monkey made of stone! Kongo, now sets out on his quest to fight evil and restore order to a desperate land. Episodes 1-3"
8,43074,Ghostbusters,2016-07-14,2016,117,en,Lançado,"Following a ghost invasion of Manhattan, paranormal enthusiasts Erin Gilbert and Abby Yates, nuclear engineer Jillian Holtzmann, and subway worker Patty Tolan band together to stop the otherworldly threat."
9,45033,20 Seconds of Joy,2018-01-01,2018,60,de,Lançado,"Traces the story of an extreme athlete, past and present; but also explores the psychology behind life, death, risk and the confrontation of fear."
10,46983,The Song of Styrene,2022-05-23,2022,13,fr,Lançado,Le chant du Styrène is a 1958 French documentary film directed by Alain Resnais. The film was an order by French industrial group Pechiney to highlight the merits of plastics.


## 2. dim_genres

Catálogo único e deduplicado de gêneros. Origem: `silver.tb_generos`.

**Decisão:** a dimensão só recebe os gêneros da lista de domínio aplicada na Silver, portanto tem 19 linhas.

In [0]:
df_dim_genres = (
    spark.table(f"{silver_schema}.tb_generos")
    .select(F.col("genero").alias("nome_genero"))

    #cada gênero aparece uma unica vez
    .distinct()
    .withColumn("sk_genre_id", F.row_number().over(Window.orderBy("nome_genero")).cast("bigint"))
    .select("sk_genre_id", "nome_genero")
)

df_dim_genres.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_genres")

display(df_dim_genres.limit(25))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sk_genre_id,nome_genero
1,Action
2,Adventure
3,Animation
4,Comedy
5,Crime
6,Documentary
7,Drama
8,Family
9,Fantasy
10,History


## 3. dim_people

Pessoas físicas envolvidas na obra (Ator, Diretor e Roteirista). Origem: `silver.tb_pessoas_empresas`.

**Decisão:** o grão é nome + tipo de atuação. Uma pessoa que atua e dirige aparece em duas linhas, uma por `tipo_pessoa`, o que mantém a relação com o filme sem ambiguidade. Produtoras ficam na `dim_companies`.

In [0]:
df_dim_people = (
    spark.table(f"{silver_schema}.tb_pessoas_empresas")

    #produtoras são empresas e ficam na dim_companies
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(
        F.col("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    )

    #a mesma pessoa pode atuar e dirigir, por isso o grão da dimensão é nome + tipo
    .distinct()
    .withColumn("sk_person_id", F.row_number().over(Window.orderBy("tipo_pessoa", "nome_pessoa")).cast("bigint"))
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)

df_dim_people.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_people")

display(df_dim_people.limit(25))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sk_person_id,nome_pessoa,tipo_pessoa
1,'ana Ika,Ator
2,'e-gotti' Eric Johnson,Ator
3,'jeeva' Ravi,Ator
4,'meesai' Mohan,Ator
5,'meesai' Rajendran,Ator
6,'om' Rakesh Chaturvedi,Ator
7,'poo' Ram,Ator
8,'sunday Jeff' Silverman,Ator
9,2 Chainz,Ator
10,2 Kupzz,Ator


## 4. dim_companies

Catálogo único de produtoras e estúdios. Origem: `silver.tb_pessoas_empresas`.

**Decisão:** empresas e pessoas ficam em dimensões separadas, pois representam entidades diferentes e são analisadas de formas diferentes (por exemplo, lucro por produtora).

In [0]:
df_dim_companies = (
    spark.table(f"{silver_schema}.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_entidade").alias("nome_produtora"))
    .distinct()
    .withColumn("sk_company_id", F.row_number().over(Window.orderBy("nome_produtora")).cast("bigint"))
    .select("sk_company_id", "nome_produtora")
)

df_dim_companies.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_companies")

display(df_dim_companies.limit(25))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sk_company_id,nome_produtora
1,#1nfluence Production
2,#beardforce Films
3,#sinning Works
4,& Extermination In An American City
5,& Space Productions
6,'s Wonderful Pictures
7,((o))eco
8,(mark Paul Wake
9,(not) Heroine Movies
10,(notice Me) Kid Vicious


## 5. dim_reviews

Avaliações dos usuários resumidas por filme. Origem: `silver.tb_avaliacoes_usuarios`.

**Decisões:**
- Uma linha por filme, com a quantidade de avaliações e a nota média arredondada para 2 casas. A quantidade conta todas as avaliações, e a média ignora as notas nulas.
- Avaliações de filmes que não existem na `dim_movies` são descartadas com `inner join`, pois não teriam uma FK válida.
- A tabela `df_lk_movies` (apoio) traduz o `id_filme` numérico da Silver para a chave substituta.

In [0]:
#tabela de apoio para trocar a chave natural pela chave substituta
df_lk_movies = (
    spark.table(f"{gold_schema}.dim_movies")
    .select(
        "sk_movie_id",
        F.col("id_filme").cast(LongType()).alias("id_filme")
    )
)

df_dim_reviews = (
    spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")

    #contagem de avaliações e média das notas (notas nulas são ignoradas pela média)
    .groupBy("id_filme")
    .agg(
        F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).alias("nota_media_usuarios")
    )

    #avaliações de filmes que não existem na dim_movies não possuem FK válida e ficam fora
    .join(df_lk_movies, on="id_filme", how="inner")
    .withColumn("sk_review_id", F.row_number().over(Window.orderBy("sk_movie_id")).cast("bigint"))
    .select(
        "sk_review_id",
        "sk_movie_id",
        "qtd_avaliacoes_usuarios",
        "nota_media_usuarios"
    )
)

df_dim_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_reviews")

display(df_dim_reviews.limit(25))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


sk_review_id,sk_movie_id,qtd_avaliacoes_usuarios,nota_media_usuarios
1,2,1,8.3
2,3,2,5.15
3,4,1,2.5
4,5,1,0.1
5,10,1,0.2
6,15,1,4.7
7,16,1,4.9
8,24,1,9.3
9,26,1,1.2
10,28,1,8.2


## 6. bridge tables

Conectam a `dim_movies` às dimensões periféricas (relação N:N) sem duplicar a fato.

**Decisões:**
- Um filme tem vários gêneros, atores e produtoras, e uma pessoa ou empresa participa de vários filmes. As pontes resolvem isso sem repetir linhas na fato.
- Todas usam `inner join` com a `dim_movies` e com a dimensão correspondente, então não existe FK nula, e `distinct` garante uma linha por par.
- A pessoa é identificada por nome + tipo, o mesmo grão da `dim_people`.

In [0]:
#bridge_movie_genre
df_bridge_movie_genre = (
    spark.table(f"{silver_schema}.tb_generos")
    .join(df_lk_movies, on="id_filme", how="inner")
    .join(
        spark.table(f"{gold_schema}.dim_genres"),
        F.col("genero") == F.col("nome_genero"),
        how="inner"
    )
    .select("sk_movie_id", "sk_genre_id")
    .distinct()
)

df_bridge_movie_genre.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.bridge_movie_genre")

#bridge_movie_person
df_bridge_movie_person = (
    spark.table(f"{silver_schema}.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .join(df_lk_movies, on="id_filme", how="inner")

    #a pessoa é identificada pelo nome e pelo tipo de atuação
    .join(
        spark.table(f"{gold_schema}.dim_people"),
        (F.col("nome_entidade") == F.col("nome_pessoa")) &
        (F.col("tipo_entidade") == F.col("tipo_pessoa")),
        how="inner"
    )
    .select("sk_movie_id", "sk_person_id")
    .distinct()
)

df_bridge_movie_person.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.bridge_movie_person")

#bridge_movie_company
df_bridge_movie_company = (
    spark.table(f"{silver_schema}.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .join(df_lk_movies, on="id_filme", how="inner")
    .join(
        spark.table(f"{gold_schema}.dim_companies"),
        F.col("nome_entidade") == F.col("nome_produtora"),
        how="inner"
    )
    .select("sk_movie_id", "sk_company_id")
    .distinct()
)

df_bridge_movie_company.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.bridge_movie_company")

display(df_bridge_movie_genre.limit(10))
display(df_bridge_movie_person.limit(10))
display(df_bridge_movie_company.limit(10))

sk_movie_id,sk_genre_id
79927,1
80585,4
80710,4
80726,11
81118,7
81137,7
81160,13
81600,13
81753,11
81776,8


sk_movie_id,sk_person_id
80230,25995
80430,189753
80593,211113
80674,216381
86516,136995
88131,146366
88280,195590
90486,200059
91026,51484
97292,162137


sk_movie_id,sk_company_id
80399,1568
81843,38506
82178,23246
83816,1094
83861,15651
85914,30180
86148,13878
87500,13470
88368,10315
92105,29787


## 7. fact_movies_performance

Grão: um registro por filme lançado. Parte da `dim_movies` e usa `left join` com financeiro e métricas, que possuem um único registro por `id_filme` na Silver, então os joins não duplicam o grão.

**Decisões:**
- Apenas filmes com status `Lançado`. Os ids sem cadastro em `tb_info_filmes` não entram (ver decisões de modelagem no início).
- `left join`: o filme permanece na fato mesmo sem dados financeiros ou de engajamento, e essas métricas ficam `NULL`.
- A célula valida o grão comparando o total de linhas com o total de filmes distintos e interrompe a execução se houver duplicidade.

In [0]:
df_fact_movies_performance = (
    spark.table(f"{gold_schema}.dim_movies")

    #a fato consolida apenas filmes lançados
    .filter(F.col("status_filme") == "Lançado")
    .select(
        "sk_movie_id",
        F.col("id_filme").cast(LongType()).alias("id_filme")
    )

    #left join para manter o filme mesmo quando não há dados financeiros ou de engajamento
    .join(spark.table(f"{silver_schema}.tb_financeiro_filmes"), on="id_filme", how="left")
    .join(spark.table(f"{silver_schema}.tb_metricas_engajamento"), on="id_filme", how="left")

    .select(
        "sk_movie_id",
        "orcamento_usd",
        "receita_usd",
        "lucro_usd",
        "orcamento_brl",
        "receita_brl",
        "lucro_brl",
        "popularidade",
        "nota_media_tmdb",
        "qtd_votos_tmdb",
        "nota_media_imdb",
        "qtd_votos_imdb"
    )
)

#validação do grão: um registro por filme
total_fato = df_fact_movies_performance.count()
filmes_distintos = df_fact_movies_performance.select("sk_movie_id").distinct().count()

if total_fato != filmes_distintos:
    raise ValueError(f"A fato possui grão duplicado: {total_fato} linhas para {filmes_distintos} filmes.")

print(f"[PASS] fact_movies_performance | {total_fato} filmes, um registro por filme")

df_fact_movies_performance.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.fact_movies_performance")

display(df_fact_movies_performance.limit(25))

[PASS] fact_movies_performance | 96463 filmes, um registro por filme


sk_movie_id,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
1,25000000.00,83080890.00,58080890.00,78682500.00,261480485.10,182797985.10,24.584,4.966,2375,null,46286
2,null,null,null,null,null,null,8.929,7.064,118,6.6,4617
3,7500000.00,null,-7500000.00,32363250.00,null,-32363250.00,null,3.161,28,null,null
4,null,null,null,null,null,null,2.98,7.1,null,7.8,212
5,90000000.00,426505244.00,336505244.00,374544000.00,1774944223.43,1400400223.43,46.619,7.139,7570,6.5,199420
6,null,null,null,null,null,null,3.403,6.63,27,6.8,1750
7,null,null,null,null,null,null,1.561,10.0,1,6.4,34
8,144000000.00,229147509.00,85147509.00,465192000.00,740261027.82,275069027.82,40.052,5.371,5976,null,260417
9,337200.00,null,-337200.00,1115255.28,null,-1115255.28,0.6,8.0,2,null,154
10,null,null,null,null,null,null,0.693,8.5,2,7.0,1209


## 8. gold_genai_movies_context

Documento de contexto por filme para o Vector Search (RAG).

Campos com chance real de vir nulos: receita e orçamento (muitos filmes sem valor), ano de lançamento, sinopse, diretor e elenco. `concat()` devolve NULL na string inteira se qualquer campo for nulo, por isso cada campo recebe um fallback com `coalesce()` antes da concatenação.

**Decisões:**
- **Todos os filmes da `dim_movies`** entram, com `left join` na fato, no elenco e nos diretores. Nenhum filme desaparece por falta de um campo.
- **Elenco:** até 5 atores por filme em ordem alfabética. A Silver não guarda a ordem de créditos, então o critério é apenas determinístico. Os diretores são todos concatenados.
- **Fallbacks:** textos como "valor não informado", "elenco não informado", "diretor não informado" e "sinopse não disponível". Textos vazios ou só com espaços também são tratados como ausentes.
- **Valores financeiros** em US$, com duas casas decimais.
- **Sinopse:** barras invertidas e aspas soltas herdadas do CSV são removidas, e o ponto final é retirado para não duplicar a pontuação do modelo de frase.
- A célula valida que nenhum documento ficou nulo e interrompe a execução em caso contrário.

In [0]:
#elenco: até 5 atores por filme em ordem alfabética
#a silver não guarda a ordem de créditos, então o critério é apenas determinístico
df_atores = (
    spark.table(f"{gold_schema}.bridge_movie_person")
    .join(spark.table(f"{gold_schema}.dim_people"), on="sk_person_id", how="inner")
    .filter(F.col("tipo_pessoa") == "Ator")
    .groupBy("sk_movie_id")
    .agg(
        F.concat_ws(
            ", ",
            F.slice(F.array_sort(F.collect_set("nome_pessoa")), 1, 5)
        ).alias("atores_principais")
    )
)

#diretores- um filme pode ter mais de um diretor
df_diretores = (
    spark.table(f"{gold_schema}.bridge_movie_person")
    .join(spark.table(f"{gold_schema}.dim_people"), on="sk_person_id", how="inner")
    .filter(F.col("tipo_pessoa") == "Diretor")
    .groupBy("sk_movie_id")
    .agg(
        F.concat_ws(
            ", ",
            F.array_sort(F.collect_set("nome_pessoa"))
        ).alias("diretor")
    )
)

df_gold_genai_movies_context = (
    spark.table(f"{gold_schema}.dim_movies")
    .select("sk_movie_id", "id_filme", "titulo", "ano_lancamento", "sinopse")

    #left join para que nenhum filme desapareça por falta de fato, elenco ou diretor
    .join(
        spark.table(f"{gold_schema}.fact_movies_performance").select("sk_movie_id", "receita_usd", "orcamento_usd"),
        on="sk_movie_id",
        how="left"
    )
    .join(df_atores, on="sk_movie_id", how="left")
    .join(df_diretores, on="sk_movie_id", how="left")

    #fallback de cada campo com coalesce 
    .withColumn(
        "titulo",
        F.coalesce(
            F.when(F.trim(F.col("titulo")) != "", F.col("titulo")),
            F.lit("Título não informado")
        )
    )
    .withColumn(
        "ano_texto",
        F.coalesce(
            F.col("ano_lancamento").cast("string"),
            F.lit("ano não informado")
        )
    )
    .withColumn(
        "receita_texto",
        F.coalesce(
            F.concat(F.lit("US$ "), F.format_number(F.col("receita_usd"), 2)),
            F.lit("valor não informado")
        )
    )
    .withColumn(
        "orcamento_texto",
        F.coalesce(
            F.concat(F.lit("US$ "), F.format_number(F.col("orcamento_usd"), 2)),
            F.lit("valor não informado")
        )
    )
    .withColumn(
        "atores_principais",
        F.coalesce(
            F.when(F.trim(F.col("atores_principais")) != "", F.col("atores_principais")),
            F.lit("elenco não informado")
        )
    )
    .withColumn(
        "diretor",
        F.coalesce(
            F.when(F.trim(F.col("diretor")) != "", F.col("diretor")),
            F.lit("diretor não informado")
        )
    )
    #remove barras invertidas e aspas soltas herdadas do escape do csv de origem
    .withColumn("sinopse", F.regexp_replace(F.col("sinopse"), r'[\\"]', ""))

    #a sinopse perde o ponto final para não duplicar a pontuação do template
    .withColumn(
        "sinopse",
        F.coalesce(
            F.when(
                F.trim(F.col("sinopse")) != "",
                F.regexp_replace(F.trim(F.col("sinopse")), r"[\.\s]+$", "")
            ),
            F.lit("sinopse não disponível")
        )
    )

    #concatenação em frase corrida, sem campos nulos
    .withColumn(
        "llm_context_document",
        F.concat(
            F.lit("O filme "), F.col("titulo"),
            F.lit(", lançado no ano de "), F.col("ano_texto"),
            F.lit(", faturou "), F.col("receita_texto"),
            F.lit(" e teve um custo de "), F.col("orcamento_texto"),
            F.lit(". Estrelado por "), F.col("atores_principais"),
            F.lit(" e dirigido por "), F.col("diretor"),
            F.lit(", o filme possui a seguinte sinopse: "), F.col("sinopse"),
            F.lit(".")
        )
    )

    .select(
        F.col("id_filme").alias("movie_id"),
        F.col("titulo").alias("title"),
        "llm_context_document"
    )
)

#validação
documentos_nulos = df_gold_genai_movies_context.filter(F.col("llm_context_document").isNull()).count()

if documentos_nulos > 0:
    raise ValueError(f"{documentos_nulos} filmes ficaram com llm_context_document nulo.")

print(f"[PASS] gold_genai_movies_context | {df_gold_genai_movies_context.count()} documentos, nenhum nulo")

df_gold_genai_movies_context.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.gold_genai_movies_context")

display(df_gold_genai_movies_context.limit(10))

[PASS] gold_genai_movies_context | 97879 documentos, nenhum nulo


movie_id,title,llm_context_document
14564,Rings,"O filme Rings, lançado no ano de 2017, faturou US$ 83,080,890.00 e teve um custo de US$ 25,000,000.00. Estrelado por Aimee Teegarden, Alex Roe, Bonnie Morgan, Chuck David Willis, Johnny Galecki e dirigido por F. Javier Gutiérrez, o filme possui a seguinte sinopse: Julia becomes worried about her boyfriend Holt when he explores the dark urban legend of a mysterious videotape said to kill the watcher seven days after viewing. She sacrifices herself to save her boyfriend and in doing so makes a horrifying discovery: there is a movie within the movie that no one has ever seen before.First you watch it. Then you die."
32471,Mixtape,"O filme Mixtape, lançado no ano de 2021, faturou valor não informado e teve um custo de valor não informado. Estrelado por Anthony Timpano, Audrey Hsieh, Gemma Brooke Allen, Jackson Rathbone, Julie Bowen e dirigido por Valerie Weiss, o filme possui a seguinte sinopse: sinopse não disponível."
38258,Grizzly II: Revenge,"O filme Grizzly II: Revenge, lançado no ano de 2020, faturou valor não informado e teve um custo de US$ 7,500,000.00. Estrelado por elenco não informado e dirigido por English, o filme possui a seguinte sinopse: All hell breaks loose when a giant grizzly."
38492,Billy Joel - Live at Yankee Stadium,"O filme Billy Joel - Live at Yankee Stadium, lançado no ano de 2022, faturou valor não informado e teve um custo de valor não informado. Estrelado por Billy Joel, David Brown, Jeffrey Jacobs, Liberty Devitto, Mark Rivera e dirigido por Jon Small, o filme possui a seguinte sinopse: Billy Joel plays his greatest hits in the Big Apple."
38700,Bad Boys for Life,"O filme Bad Boys for Life, lançado no ano de 2020, faturou US$ 426,505,244.00 e teve um custo de US$ 90,000,000.00. Estrelado por Alexander Ludwig, Charles Melton, Jacob Scipio, Joe Pantoliano, Kate Del Castillo e dirigido por Adil El Arbi, Bilall Fallah, o filme possui a seguinte sinopse: Marcus and Mike are forced to confront new threats, career changes, and midlife crises as they join the newly created elite team AMMO of the Miami police department to take down the ruthless Armando Armas, the vicious leader of a Miami drug cartel."
42018,The Horse Thief,"O filme The Horse Thief, lançado no ano de 2019, faturou valor não informado e teve um custo de valor não informado. Estrelado por Daiba, Drashi, Gaoba, Jamco Jayang, Jiji Dan e dirigido por Peicheng Pan, Zhuangzhuang Tian, o filme possui a seguinte sinopse: Devout Buddhists, Norbu and Dolma live with their young son Tashi in a clan in Tibet. Norbu is a highwayman. After Norbu is charged with stealing from the temple, he and his family are banished. Impoverished and marginalized, they can do little when their beloved son becomes ill. Tashi dies of a fever. After a second son is born, Norbu focuses his every action on keeping this child alive, seeking re-admission to the clan for his wife and child, then risking all to save them from isolation and starvation in winter."
42330,Monkey Magic,"O filme Monkey Magic, lançado no ano de 2018, faturou valor não informado e teve um custo de valor não informado. Estrelado por Dian Tao, Jihai Ma, Liu Beichen, Ye Sun e dirigido por diretor não informado, o filme possui a seguinte sinopse: Dearth Voyd, the ruler of the dark side of the universe, wants to turn the human world into a place of violence and evil!! However, a huge meteor falls upon Flower-Fruit mountain giving birth to Kongo, a monkey made of stone! Kongo, now sets out on his quest to fight evil and restore order to a desperate land. Episodes 1-3."
43074,Ghostbusters,"O filme Ghostbusters, lançado no ano de 2016, faturou US$ 229,147,509.00 e teve um custo de US$ 144,000,000.00. Estrelado por Andy García, Cecily Strong, Charles Dance, Chris Hemsworth, Kate Mckinnon e dirigido por Paul Feig, o filme possui a seguinte sinopse: Following a ghost invasion of Manhattan, paranormal enthusiasts Erin Gilbert and Abby Yates, nuclear engineer Jillian Ho

## 9. Desafio de Analytics

Consultas de negócio executadas sobre as tabelas da Gold.

**Decisões por consulta:**

- **1. Receita total em R$:** calculada sobre a fato, ou seja, filmes lançados presentes na `dim_movies`. Os ids sem cadastro (cerca de 0,21% da receita) e os filmes não lançados ficam de fora.
- **2. Filmes mais populares:** filtra as popularidades nulas. Os anos deslocados para essa coluna foram tratados na Silver, então não distorcem o ranking.
- **3. Filmes por gênero:** usa `countDistinct` por filme sobre a ponte, considerando todos os filmes da `dim_movies`.
- **4. Maiores receitas:** usa `RANK()` sobre a receita em US$ (moeda de origem) e exibe também o valor em R$. Em caso de empate na 10ª posição, mais de 10 linhas seriam retornadas.
- **Data limite:** os recortes de 2 e 5 anos partem do lançamento mais recente entre filmes lançados com data menor ou igual a hoje, pois a base tem datas futuras (até 2029), que distorceriam a janela.
- **5. Ator com mais participações (2 anos):** usa `RANK()` e exibe todos os atores empatados na primeira posição. Na execução de referência há empate real entre vários atores com 12 participações. Verificando um deles, as 12 participações são registros distintos da série "Milk & Serial" lançados em agosto de 2024.
- **6. Produtora com maior lucro (5 anos):** soma o lucro por produtora e ordena pelo lucro em US$, exibindo também o valor em R$. Como a Silver trata a ausência de orçamento como zero no cálculo do lucro, produtoras com muitos filmes sem orçamento informado podem ter o lucro superestimado.

In [0]:
#1. receita total em R$ de todos os filmes da base
#a fato contém apenas filmes lançados presentes na dim_movies, os 1.264 ids sem cadastro em tb_info_filmes ficam de fora 
display(
    spark.table(f"{gold_schema}.fact_movies_performance")
    .agg(F.sum("receita_brl").alias("receita_total_brl"))
)

receita_total_brl
662860110157.64


In [0]:
#2. os 5 filmes com maior popularidade
display(
    spark.table(f"{gold_schema}.fact_movies_performance")
    .join(spark.table(f"{gold_schema}.dim_movies"), on="sk_movie_id", how="inner")
    .filter(F.col("popularidade").isNotNull())
    .select("titulo", "popularidade")
    .orderBy(F.col("popularidade").desc())
    .limit(5)
)

titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
The Nun II,1692.778
Meg 2: The Trench,1567.273
retribution,1547.22


In [0]:
#3. quantidade de filmes por gênero, do maior para o menor volume
display(
    spark.table(f"{gold_schema}.bridge_movie_genre")
    .join(spark.table(f"{gold_schema}.dim_genres"), on="sk_genre_id", how="inner")
    .groupBy("nome_genero")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_filmes"))
    .orderBy(F.col("qtd_filmes").desc())
)

nome_genero,qtd_filmes
Drama,32286
Documentary,18996
Comedy,18624
Thriller,10274
Horror,9729
Romance,7639
Action,6049
Crime,4747
Animation,4469
Tv Movie,4079


In [0]:
#4. os 10 filmes de maior receita com a posição no ranking 
display(
    spark.table(f"{gold_schema}.fact_movies_performance")
    .join(spark.table(f"{gold_schema}.dim_movies"), on="sk_movie_id", how="inner")
    .filter(F.col("receita_usd").isNotNull())
    .withColumn("posicao_ranking", F.rank().over(Window.orderBy(F.col("receita_usd").desc())))
    .filter(F.col("posicao_ranking") <= 10)
    .select("titulo", "receita_usd", "receita_brl", "posicao_ranking")
    .orderBy("posicao_ranking")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


titulo,receita_usd,receita_brl,posicao_ranking
Avengers: Endgame,2800000000.00,11094720000.00,1
Avatar: The Way of Water,2320250281.00,12390136500.54,2
AVENGERS: INFINITY WAR,2052415039.00,7190430847.63,3
spider-man: no way home,1921847111.00,10977782882.74,4
The Lion King,1663075401.00,6227552146.58,5
Top Gun: Maverick,1488732821.00,7160804869.01,6
Barbie,1428545028.00,6856159007.38,7
The Super Mario Bros. Movie,1355725263.00,6838413799.10,8
Black Panther,1349926083.00,4429782441.36,9
Star Wars: The Last Jedi,1332698830.00,4401904235.49,10


In [0]:
#data limite dos recortes: lançamento realizado mais recente da base 
data_limite = (
    spark.table(f"{gold_schema}.dim_movies")
    .filter(
        (F.col("status_filme") == "Lançado") &
        (F.col("data_lancamento") <= F.current_date())
    )
    .agg(F.max("data_lancamento"))
    .collect()[0][0]
)

print(f"data limite dos recortes: {data_limite}")

data limite dos recortes: 2026-02-19


In [0]:
#5. ator com a maior quantidade de participações em filmes lançados nos ultimos 2 anos
#exibe todos os atores empatados na primeira posição 
display(
    spark.table(f"{gold_schema}.bridge_movie_person")
    .join(
        spark.table(f"{gold_schema}.dim_people").filter(F.col("tipo_pessoa") == "Ator"),
        on="sk_person_id",
        how="inner"
    )
    .join(
        spark.table(f"{gold_schema}.dim_movies").filter(
            (F.col("status_filme") == "Lançado") &
            (F.col("data_lancamento") > F.add_months(F.lit(data_limite), -24)) &
            (F.col("data_lancamento") <= F.lit(data_limite))
        ),
        on="sk_movie_id",
        how="inner"
    )
    .groupBy("nome_pessoa")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_participacoes"))
    .withColumn("posicao_ranking", F.rank().over(Window.orderBy(F.col("qtd_participacoes").desc())))
    .filter(F.col("posicao_ranking") == 1)
    .orderBy("nome_pessoa")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


nome_pessoa,qtd_participacoes,posicao_ranking
Adlih Torres,12,1
Andy Dubitsky,12,1
Cooper Tomlinson,12,1
Gloria Karel,12,1
John Simmonds,12,1
Jonnathon Cripple,12,1
Sterling L. Pope,12,1
Tristan Welsh,12,1


In [0]:
#6. produtora com o maior lucro nos últimos 5 anos
#ordenada pelo lucro em dolar, o lucro em R$ é exibido para conferência
display(
    spark.table(f"{gold_schema}.bridge_movie_company")
    .join(spark.table(f"{gold_schema}.dim_companies"), on="sk_company_id", how="inner")
    .join(
        spark.table(f"{gold_schema}.dim_movies").filter(
            (F.col("status_filme") == "Lançado") &
            (F.col("data_lancamento") > F.add_months(F.lit(data_limite), -60)) &
            (F.col("data_lancamento") <= F.lit(data_limite))
        ),
        on="sk_movie_id",
        how="inner"
    )
    .join(spark.table(f"{gold_schema}.fact_movies_performance"), on="sk_movie_id", how="inner")
    .groupBy("nome_produtora")
    .agg(
        F.sum("lucro_usd").alias("lucro_total_usd"),
        F.sum("lucro_brl").alias("lucro_total_brl")
    )
    .orderBy(F.col("lucro_total_usd").desc())
    .limit(5)
)

nome_produtora,lucro_total_usd,lucro_total_brl
Universal Pictures,5348017131.00,27216239853.79
Marvel Studios,4753462823.00,25526824229.92
Columbia Pictures,3564025245.00,19433295676.98
Pascal Pictures,2701952454.00,14965616280.78
Illumination,2430688795.00,12599790868.75
